# ManualMate — embed chunks on Colab and measure the baseline

Run the cells from top to bottom. Before you start:

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click the **key icon** (Secrets) on the left. Add a secret named `DATABASE_URL` with your Neon connection string (the same value as in your `.env`). Turn on **Notebook access**.
3. Have `data/chunks/chunks.jsonl` from your laptop ready to upload.

The notebook keeps nothing important: the vectors go to Neon, and the results file is downloaded to your laptop.

## 1. Check the GPU
Expected: a line with `Tesla T4`.

In [ ]:
!nvidia-smi -L

## 2. Get the latest code from GitHub

In [ ]:
import os
if not os.path.exists("/content/manualmate"):
    !git clone https://github.com/furkanhf/manualmate.git /content/manualmate
%cd /content/manualmate
!git pull
!git log --oneline -1

## 3. Install the libraries
Colab already has PyTorch with GPU support, so we do not install it.

In [ ]:
!pip install -q sentence-transformers==6.2.0 rank-bm25==0.2.2 "psycopg[binary]" pgvector python-dotenv
!pip install -q -e . --no-deps

## 4. Load the database secret
Expected: `DATABASE_URL loaded`. The value itself is never printed.

In [ ]:
from google.colab import userdata
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
print("DATABASE_URL loaded")

## 5. Upload chunks.jsonl
Click **Choose Files** and select `D:\\Projects\\manualmate\\data\\chunks\\chunks.jsonl`.
Expected: `2230 data/chunks/chunks.jsonl`.

In [ ]:
from google.colab import files
os.makedirs("data/chunks", exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
os.replace(name, "data/chunks/chunks.jsonl")
!wc -l data/chunks/chunks.jsonl

## 6. Embed all chunks and load them into Neon
Skip cells 5 and 6 if the chunks did not change (Neon keeps them).

The first run downloads bge-m3 (~2.3 GB, 1–2 min). Embedding takes about 1–2 min on a T4.
Expected last line: `Database now has 2230 chunks`.

In [ ]:
!python scripts/build_index.py

## 7. Measure retrieval (Recall@5 on 50 questions)
`routed` = parse the question → pick the manual → dense search in that manual.
- `--query-text issue`: search without brand and model words (default)
- `--query-text full`: search with the whole question
- `routed_hybrid`: the same, plus BM25 (RRF) inside the manual

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue
!python scripts/run_eval.py --method routed --query-text full
!python scripts/run_eval.py --method routed_hybrid --query-text issue

## 7b. Rerank (week 2)
`routed_rerank` = routed dense search finds the top 20 chunks → `bge-reranker-v2-m3` reads the question + each chunk together → new order.
- `--rerank-text full`: the reranker reads the whole question
- `--rerank-text issue`: the reranker reads the question without brand and model words

The first run downloads the reranker (~2.3 GB, 1–2 min). Compare with routed_issue: R@1 0.62, R@5 0.90, MRR 0.75.
Expected: `Loaded BAAI/bge-reranker-v2-m3 on cuda`, the scores, `rerank time per question`, and a list like `q033  2 -> 1`.

In [ ]:
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text full
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue

## 7c. Rerank + query rewriting (week 2)
`--rewrites` = search with the question + Gemini's manual words from `eval/rewrites.jsonl` (made on the laptop with `scripts/make_rewrites.py`, then committed). The reranker still reads the question only. Colab needs no Gemini key.
- line 1: dense only + rewrites (shows what the rewrites alone do)
- line 2: dense + rewrites + rerank

Compare with 7b (rr-issue): R@1 0.66, R@5 0.94, MRR 0.78. Expected first line: `Rewrites: 50 loaded`.

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue --rewrites
!python scripts/run_eval.py --method routed_rerank --query-text issue --rewrites

## 7d. Answers (week 3)
Full pipeline on every test question: router → dense top 20 → rerank → top 5 → Gemini → refusal gates → citation check.
Gate 1 is OFF in this run: Gemini answers every question, and the script saves the top rerank score. The summary then tries many thresholds without new Gemini calls.

Before this cell: add **GEMINI_API_KEY** to Colab Secrets (🔑 on the left). Value only: no quotes, no `GEMINI_API_KEY=`, no extra line. Turn on *Notebook access*.

Run the quick test first (3 questions per set, ~1 min). Expected: `GEMINI_API_KEY loaded`, then lines like `q001  answer ok  score=0.9...`, then `===== SUMMARY`.

In [ ]:
!pip install -q "google-genai"
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("GEMINI_API_KEY loaded")
!python scripts/run_answer_eval.py --limit 3

Full run: 50 in-scope + 30 out-of-scope + 10 clarify questions (~80 Gemini calls, about 5–8 minutes).
Then run cell 8 to download the two new files (`..._answers.jsonl` and `..._answers_summary.json`).

In [ ]:
!python scripts/run_answer_eval.py

## 8. Download the NEW results files
Only the files that are not in Git yet (the runs of this session). The old results are already on your laptop.
Save them in `D:\\Projects\\manualmate\\eval\\results\\`, then commit them. If the browser asks "Allow multiple downloads?", click Allow.

In [ ]:
from google.colab import files   # also imported in cell 5; needed when cell 5 is skipped
import subprocess

# Files in eval/results that Git does not know yet = the new runs of this session.
new = subprocess.run(["git", "ls-files", "--others", "--exclude-standard", "eval/results"],
                     capture_output=True, text=True).stdout.split()
print(f"{len(new)} new results file(s)")
for path in sorted(new):
    print(path)
    files.download(path)